# L11 · DPO: RL 없이 보이는 RL 목적함수

## Goal

- chosen/rejected log-ratio를 계산한다
- beta를 해석한다
- online RL과 차이를 설명한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L11:toy:42").hexdigest()
print(f"lesson=L11 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L11 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:34e94f892bcb390a2847aa160775bea8eeb0e49ffef2a64e7d2a6f7fe35c6cdc data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: preference data → **DPO** → offline alignment 평가

$$L_{DPO}=-\log\sigma\left(\beta\left[(\log\pi_\theta(y_w|x)-\log\pi_{ref}(y_w|x))-(\log\pi_\theta(y_l|x)-\log\pi_{ref}(y_l|x))\right]\right)$$

DPO는 chosen이 rejected보다 reference 대비 얼마나 더 좋아졌는지를 logistic loss로 학습합니다. 별도 reward model과 online rollout이 없지만 reference policy와 preference data가 암묵적인 RL 문제를 정의합니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** chosen/rejected를 바꾸면 loss가 같을까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>일반적으로 다릅니다. preference margin의 부호가 뒤집혀 sigmoid의 반대쪽을 평가합니다.</details>

In [2]:
from rl_study.algorithms.dpo import dpo_loss
policy_chosen = torch.tensor([-1.0, -0.5])
policy_rejected = torch.tensor([-2.0, -0.4])
reference_chosen = torch.tensor([-1.4, -0.7])
reference_rejected = torch.tensor([-1.8, -0.6])
dpo_output = dpo_loss(
    policy_chosen, policy_rejected, reference_chosen, reference_rejected, beta=0.2
)
swapped_output = dpo_loss(
    policy_rejected, policy_chosen, reference_rejected, reference_chosen, beta=0.2
)
print({"logits": dpo_output.logits.tolist(),
       "loss": round(float(dpo_output.loss), 4),
       "swapped_loss": round(float(swapped_output.loss), 4)})

{'logits': [0.12000000476837158, -5.9604645663569045e-09], 'loss': 0.664, 'swapped_loss': 0.724}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** sequence log-prob 합계와 평균은 length bias가 다르므로 reduction을 명시해야 합니다. IPO, KTO 등은 선호 noise와 데이터 형태에 대한 다른 가정을 둔 대안입니다.

**흔한 함정:** reference 항을 빼거나 chosen과 rejected의 순서를 뒤집어도 loss는 유한합니다. 손계산 parity와 swapped-pair test가 의미 오류를 잡습니다. 회귀 test: `test_dpo_loss_matches_hand_calculation`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert dpo_output.loss.item() != swapped_output.loss.item()
assert torch.isfinite(dpo_output.loss)
print("checks=passed")

checks=passed


**회상 문제:** DPO에서 reference model을 제거하면 어떤 기준점이 사라지나요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** 원래 pair loss 0.664와 swapped loss 0.724가 달랐습니다. 출력된 두 margin 중 하나는 거의 0이라 그 sample은 약한 학습 신호를 줍니다.
- 실제 확인: `test_dpo_loss_matches_hand_calculation`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L12에서 한 prompt의 여러 rollout reward를 group 안에서 상대화하는 GRPO로 돌아갑니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

[상세 구현 문서](../../docs/algorithms/dpo.md) · [강좌 지도](../../docs/course-map.md)

## Sources

- `dpo-2023` — `docs/sources.yml`
- `repo-dpo` — `docs/sources.yml`